In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("Module4_ChurnPrediction") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [4]:
import glob

folder_path = "E:/Ky2_2025_2026/BigData/CLEARDATA/*.csv"
file_list = glob.glob(folder_path)

print(f"Đã tìm thấy {len(file_list)} file CSV trong thư mục.")

if len(file_list) > 0:
    print("Đang đọc toàn bộ dữ liệu...")
    df = spark.read.csv(file_list, header=True, inferSchema=False)
    
    print("--- Cấu trúc dữ liệu ---")
    df.printSchema()
    
    print("--- 5 dòng đầu tiên ---")
    df.show(5)
else:
    print("Không tìm thấy file CSV nào, bạn kiểm tra lại đường dẫn nhé!")

Đã tìm thấy 29 file CSV trong thư mục.
Đang đọc toàn bộ dữ liệu...
--- Cấu trúc dữ liệu ---
root
 |-- user_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- recording_msid: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- release_name: string (nullable = true)

--- 5 dòng đầu tiên ---
+-------+-------------------+--------------------+--------------------+------------+--------------------+
|user_id|          timestamp|      recording_msid|          track_name| artist_name|        release_name|
+-------+-------------------+--------------------+--------------------+------------+--------------------+
|      1|2026-01-31 12:57:58|2d39940b-f575-42a...|               Auxin|        Cell|      Onwards System|
|      1|2026-01-31 13:04:37|230a1f84-609d-43d...|             Figment|        Cell|      Onwards System|
|      1|2026-01-31 13:11:07|025df4fb-5bc4-4b3...|              Geiger|        Cell|      Onw

In [ ]:
# from pyspark.sql.functions import col, to_timestamp, max as spark_max, count, countDistinct, datediff, lit, when

# df_filtered = df.filter(col("timestamp").rlike("^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))
# df_clean = df_filtered.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
# df_clean = df_clean.dropna(subset=["timestamp", "user_id", "recording_msid"])

# latest_date_row = df_clean.select(spark_max("timestamp")).collect()[0][0]
# print(f"Mốc thời gian hiện tại (dòng log mới nhất): {latest_date_row}")

# user_features = df_clean.groupBy("user_id").agg(
#     spark_max("timestamp").alias("last_listen_date"),   
#     count("recording_msid").alias("total_listens"),         
#     countDistinct("artist_name").alias("unique_artists")    
# )

# user_profile = user_features.withColumn(
#     "days_inactive",
#     datediff(lit(latest_date_row), col("last_listen_date"))
# ).withColumn(
#     "is_churn",
#     when(col("days_inactive") >= 14, 1).otherwise(0)
# )

# print("--- 5 dòng dữ liệu User Profile đầu tiên ---")
# user_profile.show(5)

# print("--- Thống kê số lượng User (Churn vs Active) ---")
# user_profile.groupBy("is_churn").count().show()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21944\3139802146.py:3: SyntaxWarning: invalid escape sequence '\d'
  df_filtered = df.filter(col("timestamp").rlike("^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))


Mốc thời gian hiện tại (dòng log mới nhất): 2026-02-05 00:03:33
--- 5 dòng dữ liệu User Profile đầu tiên ---
+-------+-------------------+-------------+--------------+-------------+--------+
|user_id|   last_listen_date|total_listens|unique_artists|days_inactive|is_churn|
+-------+-------------------+-------------+--------------+-------------+--------+
|  19338|2026-02-04 20:22:22|         2729|           897|            1|       0|
|  28334|2026-02-04 23:58:22|         4803|          2384|            1|       0|
|  38672|2026-02-04 21:26:51|         4597|          1025|            1|       0|
|  48063|2026-02-04 23:28:06|          104|            53|            1|       0|
|  91959|2026-02-04 22:42:22|       113961|          9376|            1|       0|
+-------+-------------------+-------------+--------------+-------------+--------+
only showing top 5 rows
--- Thống kê số lượng User (Churn vs Active) ---
+--------+-----+
|is_churn|count|
+--------+-----+
|       1| 2431|
|       0|20

In [ ]:
# from pyspark.ml.feature import VectorAssembler
# from pyspark.ml.classification import RandomForestClassifier
# from pyspark.ml.evaluation import BinaryClassificationEvaluator
# from pyspark.sql.functions import col

# feature_columns = ["total_listens", "unique_artists"]

# assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# ml_data = assembler.transform(user_profile).select(
#     col("user_id"), 
#     col("features"), 
#     col("is_churn").alias("label")
# )

# train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)
# print(f"Số lượng data huấn luyện (Train): {train_data.count()}")
# print(f"Số lượng data kiểm thử (Test): {test_data.count()}")

# rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=5)

# print("Đang huấn luyện mô hình Random Forest, vui lòng chờ...")
# rf_model = rf.fit(train_data)
# print("Huấn luyện xong!")

# predictions = rf_model.transform(test_data)
# print("--- Kết quả dự đoán trên tập Test (5 người đầu tiên) ---")

# predictions.select("user_id", "label", "prediction", "probability").show(5, truncate=False)

# evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
# auc = evaluator.evaluate(predictions)
# print(f"Độ chính xác của mô hình (AUC-ROC): {auc:.4f}")

Số lượng data huấn luyện (Train): 18818
Số lượng data kiểm thử (Test): 4558
Đang huấn luyện mô hình Random Forest, vui lòng chờ...
Huấn luyện xong!
--- Kết quả dự đoán trên tập Test (5 người đầu tiên) ---
+-------+-----+----------+----------------------------------------+
|user_id|label|prediction|probability                             |
+-------+-----+----------+----------------------------------------+
|10058  |0    |0.0       |[0.8696640327383758,0.13033596726162425]|
|1043   |0    |0.0       |[0.9834902779083389,0.01650972209166108]|
|10512  |0    |0.0       |[0.5354691931614688,0.46453080683853115]|
|10803  |0    |0.0       |[0.980573681927777,0.019426318072223038]|
|11037  |0    |0.0       |[0.7100935000176535,0.28990649998234647]|
+-------+-----+----------+----------------------------------------+
only showing top 5 rows
Độ chính xác của mô hình (AUC-ROC): 0.8815


In [ ]:
# import pandas as pd

# print("Đang chuyển đổi dữ liệu sang Pandas để xuất file...")

# pandas_df = web_data.toPandas()

# print("--- 5 dòng dữ liệu chuẩn bị đưa lên Web ---")
# print(pandas_df.head())

# output_path = "E:/Ky2_2025_2026/BigData/web_dashboard_data.csv" 
# pandas_df.to_csv(output_path, index=False)

# print(f"Đã xuất file dữ liệu Web thành công tại: {output_path}")

Đang chuyển đổi dữ liệu sang Pandas để xuất file...
--- 5 dòng dữ liệu chuẩn bị đưa lên Web ---
  user_id  total_listens  unique_artists  churn_risk_percent
0   19338           2729             897                1.53
1   28334           4803            2384                1.53
2   38672           4597            1025                1.53
3   48063            104              53                9.04
4   91959         113961            9376                1.53
Đã xuất file dữ liệu Web thành công tại: E:/Ky2_2025_2026/BigData/web_dashboard_data.csv


### Mô hình Baseline (Data Leakage)

In [ ]:
# import glob
# import pandas as pd
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col, to_timestamp, max as spark_max, min as spark_min, count, countDistinct, datediff, lit, when, hour, sum as spark_sum, round
# from pyspark.ml.feature import VectorAssembler
# from pyspark.ml.classification import RandomForestClassifier
# from pyspark.ml.evaluation import BinaryClassificationEvaluator
# from pyspark.ml.functions import vector_to_array

# print("--- KHỞI ĐỘNG HỆ THỐNG & NẠP DỮ LIỆU ---")
# spark = SparkSession.builder \
#     .appName("Module4_ChurnPrediction") \
#     .config("spark.driver.memory", "4g") \
#     .getOrCreate()

# folder_path = "E:/Ky2_2025_2026/BigData/CLEARDATA/*.csv"
# file_list = glob.glob(folder_path)
# df = spark.read.csv(file_list, header=True, inferSchema=False)

# df_filtered = df.filter(col("timestamp").rlike("^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))
# df_clean = df_filtered.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
# df_clean = df_clean.dropna(subset=["timestamp", "user_id", "recording_msid"])

# print("--- PHẦN 1: TÍNH TOÁN ĐẶC TRƯNG NÂNG CAO ---")
# df_enhanced = df_clean.withColumn("hour", hour(col("timestamp"))) \
#                       .withColumn("is_night", when((col("hour") >= 22) | (col("hour") <= 5), 1).otherwise(0))

# latest_date_row = df_enhanced.select(spark_max("timestamp")).collect()[0][0]

# user_features = df_enhanced.groupBy("user_id").agg(
#     spark_max("timestamp").alias("last_listen_date"),
#     spark_min("timestamp").alias("first_listen_date"),
#     count("recording_msid").alias("total_listens"),
#     countDistinct("artist_name").alias("unique_artists"),
#     spark_sum("is_night").alias("night_listens")
# )

# user_profile_adv = user_features.withColumn(
#     "tenure_days", datediff(col("last_listen_date"), col("first_listen_date"))
# ).withColumn(
#     "tenure_days", when(col("tenure_days") == 0, 1).otherwise(col("tenure_days")) # Tránh lỗi chia cho 0
# ).withColumn(
#     "daily_listen_rate", round(col("total_listens") / col("tenure_days"), 2)
# ).withColumn(
#     "night_listen_ratio", round(col("night_listens") / col("total_listens"), 4)
# ).withColumn(
#     "artist_diversity", round(col("unique_artists") / col("total_listens"), 4)
# ).withColumn(
#     "days_inactive", datediff(lit(latest_date_row), col("last_listen_date"))
# ).withColumn(
#     "is_churn", when(col("days_inactive") >= 14, 1).otherwise(0)
# )

# user_profile_adv.cache()

# print("--- PHẦN 2: HUẤN LUYỆN MÔ HÌNH ---")
# feature_cols = ["total_listens", "daily_listen_rate", "night_listen_ratio", "artist_diversity", "tenure_days"]
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# full_ml_data = assembler.transform(user_profile_adv)
# train_data, test_data = full_ml_data.randomSplit([0.8, 0.2], seed=42)

# rf = RandomForestClassifier(featuresCol="features", labelCol="is_churn", numTrees=50, maxDepth=5)
# rf_model = rf.fit(train_data)

# predictions = rf_model.transform(test_data)
# evaluator = BinaryClassificationEvaluator(labelCol="is_churn", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
# auc = evaluator.evaluate(predictions)
# print(f"Độ chính xác AUC-ROC với Features mới: {auc:.4f}")

# print("--- PHẦN 3: XUẤT DỮ LIỆU DASHBOARD ---")
# all_predictions = rf_model.transform(full_ml_data)

# final_results = all_predictions.withColumn(
#     "churn_risk_percent", round(vector_to_array(col("probability"))[1] * 100, 2)
# )

# web_data = final_results.select(
#     "user_id", "total_listens", "daily_listen_rate", "night_listen_ratio", 
#     "artist_diversity", "tenure_days", "churn_risk_percent"
# )

# pandas_df = web_data.toPandas()
# output_path = "E:/Ky2_2025_2026/BigData/web_dashboard_data_v2.csv"
# pandas_df.to_csv(output_path, index=False)

# print(f"Hoàn thành! Đã lưu file tại: {output_path}")

<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3900\2305625042.py:20: SyntaxWarning: invalid escape sequence '\d'
  df_filtered = df.filter(col("timestamp").rlike("^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))


--- KHỞI ĐỘNG HỆ THỐNG & NẠP DỮ LIỆU ---
--- PHẦN 1: TÍNH TOÁN ĐẶC TRƯNG NÂNG CAO ---
--- PHẦN 2: HUẤN LUYỆN MÔ HÌNH ---
Độ chính xác AUC-ROC với Features mới: 0.9370
--- PHẦN 3: XUẤT DỮ LIỆU DASHBOARD ---
Hoàn thành! Đã lưu file tại: E:/Ky2_2025_2026/BigData/web_dashboard_data_v2.csv


### Mô hình time-window 

In [6]:
from pyspark.sql.functions import col, lit, when, count, countDistinct, sum as spark_sum, max as spark_max, min as spark_min, datediff, round, hour
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("--- KHỞI ĐỘNG HỆ THỐNG TIME-WINDOW ---")

# 1. ĐỊNH NGHĨA CỬA SỔ THỜI GIAN
cutoff_date = "2026-01-24 23:59:59"

# Cửa sổ quan sát (Chỉ lấy data quá khứ để tính Feature)
df_obs = df_clean.filter(col("timestamp") <= cutoff_date)

# Cửa sổ dự đoán (Chỉ lấy data tương lai để làm Label)
df_pred = df_clean.filter(col("timestamp") > cutoff_date)

print("--- PHẦN 1: TÍNH TOÁN ĐẶC TRƯNG TỪ QUÁ KHỨ ---")
# Thêm cột giờ và đêm vào df_obs
df_obs = df_obs.withColumn("hour", hour(col("timestamp"))) \
               .withColumn("is_night", when((col("hour") >= 22) | (col("hour") <= 5), 1).otherwise(0))

# Gom nhóm tính toán y hệt code cũ của bạn, nhưng TUYỆT ĐỐI CHỈ TRÊN df_obs
user_features = df_obs.groupBy("user_id").agg(
    spark_max("timestamp").alias("last_listen_date"),
    spark_min("timestamp").alias("first_listen_date"),
    count("recording_msid").alias("total_listens"),
    countDistinct("artist_name").alias("unique_artists"),
    spark_sum("is_night").alias("night_listens")
)

# Tính các chỉ số nâng cao
user_profile_adv = user_features.withColumn(
    "tenure_days", datediff(col("last_listen_date"), col("first_listen_date"))
).withColumn(
    "tenure_days", when(col("tenure_days") == 0, 1).otherwise(col("tenure_days")) # Tránh chia 0
).withColumn(
    "daily_listen_rate", round(col("total_listens") / col("tenure_days"), 2)
).withColumn(
    "night_listen_ratio", round(col("night_listens") / col("total_listens"), 4)
).withColumn(
    "artist_diversity", round(col("unique_artists") / col("total_listens"), 4)
)

print("--- PHẦN 2: TẠO NHÃN CHURN THỰC TẾ ---")
# Lấy danh sách user CÓ xuất hiện trong tuần cuối (Nghĩa là KHÔNG CHURN)
active_users = df_pred.select("user_id").distinct().withColumn("is_active", lit(1))

# Ghép với tập Feature
final_df = user_profile_adv.join(active_users, on="user_id", how="left")

# Logic Churn: Nếu is_active bị Null (không nghe bài nào trong tuần cuối) -> is_churn = 1
final_df = final_df.withColumn("is_churn", when(col("is_active").isNull(), 1).otherwise(0))
final_df = final_df.drop("is_active")

# Lưu Cache để chạy Model nhanh hơn
final_df.cache()
print(f"Tổng số user hợp lệ để huấn luyện: {final_df.count()}")

print("--- PHẦN 3: HUẤN LUYỆN MÔ HÌNH CHUẨN ---")
feature_cols = ["total_listens", "daily_listen_rate", "night_listen_ratio", "artist_diversity", "tenure_days"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

ml_data = assembler.transform(final_df)
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(featuresCol="features", labelCol="is_churn", numTrees=50, maxDepth=5)
rf_model = rf.fit(train_data)

predictions = rf_model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="is_churn", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

print(f"✅ ĐỘ CHÍNH XÁC THỰC TẾ CỦA MÔ HÌNH (AUC-ROC): {auc:.4f}")

--- KHỞI ĐỘNG HỆ THỐNG TIME-WINDOW ---
--- PHẦN 1: TÍNH TOÁN ĐẶC TRƯNG TỪ QUÁ KHỨ ---
--- PHẦN 2: TẠO NHÃN CHURN THỰC TẾ ---
Tổng số user hợp lệ để huấn luyện: 21485
--- PHẦN 3: HUẤN LUYỆN MÔ HÌNH CHUẨN ---
✅ ĐỘ CHÍNH XÁC THỰC TẾ CỦA MÔ HÌNH (AUC-ROC): 0.9113


In [7]:
from pyspark.ml.functions import vector_to_array

print("--- PHẦN 4: XUẤT DỮ LIỆU CẬP NHẬT RA DASHBOARD ---")

# 1. Chạy dự đoán trên TOÀN BỘ tập dữ liệu hợp lệ (ml_data) bằng mô hình chuẩn
all_predictions = rf_model.transform(ml_data)

# 2. Bóc tách mảng Probability thành số % nguy cơ rời bỏ dễ đọc
final_results = all_predictions.withColumn(
    "churn_risk_percent", round(vector_to_array(col("probability"))[1] * 100, 2)
)

# 3. Chọn lọc các cột đặc trưng quan trọng nhất để mang lên Web
web_data = final_results.select(
    "user_id", "total_listens", "daily_listen_rate", 
    "night_listen_ratio", "artist_diversity", "tenure_days", "churn_risk_percent"
)

# 4. CHUYỂN ĐỔI SANG PANDAS VÀ XUẤT FILE (Đoạn code của bạn)
import pandas as pd

print("Đang chuyển đổi dữ liệu sang Pandas để xuất file...")
pandas_df = web_data.toPandas()

print("--- 5 dòng dữ liệu chuẩn bị đưa lên Web ---")
print(pandas_df.head())

# Tôi thêm _v2 vào tên file để bạn lưu phiên bản mới mà không đè mất file cũ
output_path = "E:/Ky2_2025_2026/BigData/web_dashboard_data_v2.csv" 
pandas_df.to_csv(output_path, index=False)

print(f"✅ Đã xuất file dữ liệu Web thành công tại: {output_path}")

--- PHẦN 4: XUẤT DỮ LIỆU CẬP NHẬT RA DASHBOARD ---
Đang chuyển đổi dữ liệu sang Pandas để xuất file...
--- 5 dòng dữ liệu chuẩn bị đưa lên Web ---
  user_id  total_listens  daily_listen_rate  night_listen_ratio  \
0   14204            642              35.67              0.3380   
1   14899           1718             101.06              0.5501   
2   27248            546              32.12              0.0916   
3   28334           2778             163.41              0.2275   
4   40740            824              48.47              0.1517   

   artist_diversity  tenure_days  churn_risk_percent  
0            0.3287           18                2.34  
1            0.2864           17                2.34  
2            0.3516           17                2.34  
3            0.5083           17                2.34  
4            0.5498           17                2.34  
✅ Đã xuất file dữ liệu Web thành công tại: E:/Ky2_2025_2026/BigData/web_dashboard_data_v2.csv
